# Clase 3 · Pre-clase — ETL: Extracción, Transformación y Carga

**Qué haces antes de venir a clase (60-90 min):**
1. Lee este notebook y ejecuta las celdas.
2. Al terminar habrás construido una dimensión completa de la bodega de Saber 11.
3. Responde las reflexiones 🟡 y sube el notebook a Moodle.

**Objetivo concreto:** al terminar puedes describir las 3 fases del ETL y construir una tabla dimensional con clave surrogate.

## ¿Qué es ETL?

**ETL** significa **Extraer, Transformar y Cargar**. Es el proceso que mueve datos desde su fuente original hacia una bodega de datos analítica. Primero se *extrae* la información cruda de la fuente (un CSV, una API, una BD transaccional). Luego se *transforma*: se limpian los datos, se corrigen tipos y se construyen las tablas que necesita el análisis. Finalmente se *carga* el resultado en la base de datos destino.

**Analogía:** piensa en lavar y doblar ropa antes de guardarla en el armario. No guardas la ropa sucia directamente (extraer ≠ cargar sin más); primero la lavas (transformar) y luego la organizas en los cajones correctos (cargar).

## Fase E — Extracción

La extracción lee los datos **tal como están** en la fuente, sin modificarlos. Una buena práctica al leer CSVs grandes es cargar todo como texto (`dtype=str`) para no perder información. Así evitamos que pandas "adivine" tipos incorrectos — por ejemplo, que convierta el código de municipio `"05001"` al entero `5001`, perdiendo el cero inicial.

La transformación de tipos ocurre después, de forma explícita y controlada.

In [6]:
import pandas as pd

df_raw = pd.read_csv("../../datos/saber11_muestra_500k.csv", dtype=str)
print(f"Extraídos: {len(df_raw):,} registros × {len(df_raw.columns)} columnas")
df_raw[["PERIODO", "COLE_NATURALEZA", "COLE_JORNADA", "PUNT_GLOBAL"]].head(5)

Extraídos: 500,000 registros × 22 columnas


,PERIODO,COLE_NATURALEZA,COLE_JORNADA,PUNT_GLOBAL
0,20194,OFICIAL,COMPLETA,194
1,20224,OFICIAL,COMPLETA,199
2,20214,NO OFICIAL,NOCHE,268
3,20204,OFICIAL,TARDE,365
4,20204,OFICIAL,TARDE,355


### 🟡 Reflexión 1

¿Por qué leer todo como `dtype=str` en vez de dejar que pandas adivine los tipos? Menciona al menos 2 casos donde adivinar mal causaría un problema.

_Tu respuesta:_

## Fase T — Transformación (construyendo UNA dimensión)

En una bodega de datos hay varias **tablas dimensionales** que describen el contexto de los hechos. Para Saber 11 tenemos dimensiones de colegio, tiempo y geografía. Hoy construiremos **solo `dim_colegio`** para entender el proceso paso a paso. En el laboratorio construirán todas.

### ¿Qué es una clave surrogate?

Una **clave surrogate** (o subrogada) es un identificador artificial que generamos nosotros — un número entero secuencial — en lugar de usar el identificador que viene en los datos originales (clave natural). ¿Por qué no usar `COLE_NATURALEZA` directamente como clave de referencia?

- **Puede cambiar**: si la fuente corrige un valor (ej. `"OFICIAL"` → `"PÚBLICO"`), todas las referencias en el hecho quedarían rotas.
- **Puede tener NULLs**: datos históricos incompletos romperían la integridad referencial.
- **No la controlamos**: el sistema OLTP fuente puede cambiar sus convenciones sin avisarnos.

La surrogate key nos da estabilidad: es nuestra clave, la controlamos, nunca cambia.

In [7]:
df = df_raw.copy()

# Normalizar texto
for col in ["COLE_NATURALEZA", "COLE_JORNADA", "COLE_CALENDARIO", "COLE_BILINGUE"]:
    df[col] = df[col].str.strip().str.upper()

# Construir dimensión: combinaciones únicas + clave surrogate
dim_colegio = (
    df[["COLE_NATURALEZA", "COLE_JORNADA", "COLE_CALENDARIO", "COLE_BILINGUE"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_colegio.insert(0, "colegio_id", dim_colegio.index + 1)

print(f"dim_colegio: {len(dim_colegio)} combinaciones únicas")
dim_colegio

dim_colegio: 40 combinaciones únicas


,colegio_id,COLE_NATURALEZA,COLE_JORNADA,COLE_CALENDARIO,COLE_BILINGUE
0,1,OFICIAL,COMPLETA,A,N
1,2,NO OFICIAL,NOCHE,A,N
2,3,OFICIAL,TARDE,A,N
3,4,OFICIAL,MAÑANA,A,N
4,5,NO OFICIAL,TARDE,A,N
5,6,NO OFICIAL,COMPLETA,A,N
6,7,NO OFICIAL,MAÑANA,A,N
7,8,OFICIAL,SABATINA,A,N
8,9,NO OFICIAL,TARDE,B,N
9,10,OFICIAL,TARDE,A,S


### 🟡 Reflexión 2

La `dim_colegio` que construiste tiene `COLE_NATURALEZA`, `COLE_JORNADA`, `COLE_CALENDARIO` y `COLE_BILINGUE`. ¿Qué pasa con los atributos del colegio que NO están ahí (por ejemplo, el nombre del colegio o el municipio donde está)? ¿Deberían estar en esta misma tabla o en otra? ¿Por qué?

_Tu respuesta:_

## Fase L — Carga

La carga escribe los datos transformados en la base de datos destino. Con pandas usamos `to_sql`, que conecta un DataFrame directamente con SQLite (o cualquier BD compatible con SQLAlchemy).

El parámetro `if_exists` controla qué pasa si la tabla ya existe:
- `'replace'`: borra la tabla existente y la recrea desde cero. Útil para cargas completas.
- `'append'`: agrega filas a la tabla sin borrarla. Útil para cargas incrementales.
- `'fail'`: lanza un error si la tabla ya existe. Protege contra sobreescrituras accidentales.

In [8]:
import sqlite3

conn = sqlite3.connect("../../datos/saber11_bodega.db")
dim_colegio.to_sql("dim_colegio", conn, if_exists="replace", index=False)

# Verificar
n = conn.execute("SELECT COUNT(*) FROM dim_colegio").fetchone()[0]
print(f"dim_colegio en BD: {n} filas ✓")
conn.close()

dim_colegio en BD: 40 filas ✓


### 🟡 Reflexión 3

Acabas de hacer un ETL completo para UNA dimensión. En el laboratorio van a construir las 3 dimensiones restantes (`dim_tiempo`, `dim_geografia`, `dim_estudiante`) y la tabla de hechos. Antes de llegar: ¿Cuáles columnas de Saber 11 asignarías a cada dimensión? Haz un listado rápido.

**Pista para `dim_tiempo`:** el campo `PERIODO` tiene formato `AAAAQ` (ejemplo: `20254` = año 2025, semestre/quarter 4). En el laboratorio lo van a partir en dos columnas: `anio` y `quarter`. ¿Qué ventaja da eso frente a guardar `20254` como un solo número?

_Tu respuesta:_

## El hecho: midiendo lo que pasó

Ya tienes `dim_colegio` que clasifica los colegios. Ahora necesitas la tabla de hechos: la que registra **cada resultado** de estudiante y lo conecta con sus dimensiones.

En el modelo estrella de Saber 11:
- La **tabla de hechos** (`hecho_resultados`) tiene las **medidas numéricas** (puntajes) y las **claves de referencia** (FK) hacia las dimensiones.
- Cada fila del hecho = un estudiante en un período.

```
              dim_tiempo (pendiente)
                    ↑
dim_colegio  ←  hecho_resultados  →  dim_geografia (pendiente)
(colegio_id)    (colegio_id (ref.),
                 tiempo_id (ref.),
                 geo_id (ref.),
                 PUNT_*)
```

Por ahora construiremos un hecho **parcial**: solo con `colegio_id` (la única FK que ya tenemos) + los 6 puntajes. En el laboratorio añadirán las FKs de tiempo y geografía.

### Nota importante: las bodegas de datos NO usan restricciones FK

En bases de datos transaccionales (OLTP) se acostumbra declarar `FOREIGN KEY ... REFERENCES` para que la BD rechace cualquier fila que no tenga una dimensión correspondiente.

**En bodegas de datos (OLAP) esto NO se hace.** Las razones son:

1. **Datos tardíos:** los registros del hecho pueden llegar antes que su dimensión correspondiente.
2. **Calidad imperfecta:** los sistemas fuente a veces envían códigos desconocidos o vacíos.
3. **Rendimiento:** validar FK en cargas masivas de millones de filas es costoso.
4. **Patrón "Desconocido":** en vez de NULL en el hecho, se agrega una fila especial en cada dimensión con `id = -1` y atributo `"Desconocido"`. Así el hecho siempre tiene un valor válido aunque la dimensión no reconozca ese registro.

En nuestro código, `to_sql` simplemente carga los DataFrames como tablas — nunca declara `FOREIGN KEY`. La relación existe en el diseño y en las consultas (`JOIN`), pero **no está forzada por la BD**.

### 🟡 Reflexión 4

Antes de ver el código: mira las columnas del CSV de Saber 11.
- ¿Cuáles irían en la tabla de **hechos** (medidas numéricas) y cuáles en las **dimensiones** (atributos descriptivos)?
- ¿Qué columna usarías para identificar el "período" del examen? ¿Ya existe o hay que crearla?

_Tu respuesta:_

In [ ]:
# Construir hecho_resultados parcial (solo con colegio_id por ahora)
# En el lab agregarán tiempo_id y geo_id

# Unir df con dim_colegio para obtener la FK
cols_join = ["COLE_NATURALEZA", "COLE_JORNADA", "COLE_CALENDARIO", "COLE_BILINGUE"]
df_hecho = df.merge(dim_colegio, on=cols_join, how="left")

cols_puntaje = ["PUNT_LECTURA_CRITICA", "PUNT_MATEMATICAS", "PUNT_C_NATURALES",
                "PUNT_SOCIALES_CIUDADANAS", "PUNT_INGLES", "PUNT_GLOBAL"]
hecho_resultados = df_hecho[["colegio_id"] + cols_puntaje].copy()

print(f"hecho_resultados: {len(hecho_resultados):,} filas (esperado {len(df):,})")
assert len(hecho_resultados) == len(df), "¡Pérdida de filas en el merge!"
hecho_resultados.head(3)

In [ ]:
# Cargar el hecho parcial a SQLite
conn = sqlite3.connect("../../datos/saber11_bodega.db")
hecho_resultados.to_sql("hecho_resultados", conn, if_exists="replace", index=False)

# Verificar tablas en la BD
print("Tablas en la bodega:")
for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall():
    n = conn.execute(f"SELECT COUNT(*) FROM {row[0]}").fetchone()[0]
    print(f"  {row[0]}: {n:,} filas")

In [ ]:
# Una consulta de prueba: promedio de PUNT_GLOBAL por COLE_NATURALEZA
import pandas as pd

query = """
SELECT c.COLE_NATURALEZA,
       ROUND(AVG(h.PUNT_GLOBAL), 1) AS prom_global,
       COUNT(*) AS n_estudiantes
FROM hecho_resultados h
JOIN dim_colegio c ON h.colegio_id = c.colegio_id
GROUP BY c.COLE_NATURALEZA
ORDER BY prom_global DESC
"""
pd.read_sql(query, conn)

### 🟡 Reflexión 5

La consulta anterior usa la bodega (hecho + dimensión). Pero la bodega todavía está **incompleta**: faltan `dim_tiempo` y `dim_geografia`.

¿Qué consultas NO podemos hacer todavía con la bodega parcial? Nombra 2 preguntas de negocio que requieran esas dimensiones faltantes.

_Tu respuesta:_

---
**En el laboratorio construirán `dim_tiempo` y `dim_geografia`, actualizarán el hecho con las 3 claves de referencia, y ejecutarán consultas más ricas.**